# Exercise 2 — generate_project_plan and default_phases

A plan is the difference between a project that ships and one that drifts forever. `generate_project_plan` produces a Markdown document with the spec summary, deliverable checklist, and the six standard build phases. `default_phases` creates the six `Phase` objects that power the tracking system.

In [ ]:
import datetime
from dataclasses import dataclass, field

@dataclass
class CapstoneSpec:
    name: str; tagline: str; domain: str; description: str
    sections_used: list; deliverables: list; tech_stack: list

@dataclass
class Phase:
    name: str; tasks: list; done: bool = False

@dataclass
class CapstoneReport:
    spec: CapstoneSpec
    phases: list = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.date.today().isoformat())
    completed_at: str = ""

_SPEC = CapstoneSpec(
    name          = "AI Trading Bot",
    tagline       = "Paper-trading bot with sentiment and technical signals.",
    domain        = "finance",
    description   = "End-to-end AI trading bot that fetches OHLCV data, computes "
                    "technical indicators, scores news sentiment with an LLM, applies "
                    "risk controls, and runs a daily paper-trading loop with logging.",
    sections_used = [
        "Section 3: Data & Analysis (pandas, SQLite)",
        "Section 4: Real Apps (FastAPI endpoint)",
        "Section 6: AI Agents (scheduling loop)",
        "Section 7: Finance & Trading (backtester, risk manager, paper trader)",
    ],
    deliverables  = [
        "paper_trader.py with buy/sell/portfolio_value",
        "bot_runner.py with daily scheduling and logging",
        "risk.py with stop-loss and drawdown controls",
        "Deployed FastAPI endpoint",
        "Portfolio case study",
    ],
    tech_stack    = ["Python", "pandas", "Ollama", "SQLite", "FastAPI"],
)
def validate_spec(spec):
    errors = []
    for field_name in ("name", "tagline", "domain", "description"):
        val = getattr(spec, field_name, "")
        if not isinstance(val, str) or not val.strip():
            errors.append(f"{field_name} must be a non-empty string")
    if len(spec.deliverables) < 3:
        errors.append(f"at least 3 deliverables required, got {len(spec.deliverables)}")
    if len(spec.tech_stack) < 2:
        errors.append(f"at least 2 tech stack items required, got {len(spec.tech_stack)}")
    if not spec.sections_used:
        errors.append("sections_used must reference at least one course section")
    return len(errors) == 0, errors

def generate_project_plan(spec):
    """Generate a Markdown implementation plan.

    Must contain:
      - '# {spec.name}' at the start (or in title)
      - '## Overview' section with description
      - '## Deliverables' with each deliverable as '- [ ] item'
      - '## Build Phases' with 6 phase headings (Phase 1 through Phase 6)

    Returns:
        str — Markdown
    """
    deliverable_list = "\n".join(f"- [ ] {d}" for d in spec.deliverables)
    tech_list        = "\n".join(f"- {t}"     for t in spec.tech_stack)
    sections_list    = "\n".join(f"- {s}"     for s in spec.sections_used)
    # TODO: assemble the Markdown plan
    return ""


def default_phases(spec):
    """Return the 6 standard build phases with tasks.

    Returns list[Phase]: Plan, Build, Test, Deploy, Document, Share.
    Each phase has 3 tasks. done=False for all.
    """
    ai_backend = next(
        (t for t in spec.tech_stack if t.lower() in ("ollama", "llama", "llamacpp")),
        "AI backend",
    )
    # TODO: return list of 6 Phase objects
    return []


### Checks

In [ ]:
checks = 0

# 1 — generate_project_plan: contains project name
try:
    plan = generate_project_plan(_SPEC)
    assert isinstance(plan, str) and len(plan) > 100
    assert _SPEC.name in plan, f"name '{_SPEC.name}' not in plan"
    checks += 1; print(f"✅ 1 plan contains project name '{_SPEC.name}'")
except Exception as e:
    print("❌ 1:", e)

# 2 — plan: ## Overview with description
try:
    plan = generate_project_plan(_SPEC)
    assert "## Overview" in plan, "missing ## Overview"
    assert _SPEC.description[:30] in plan, "description not in plan"
    checks += 1; print("✅ 2 plan has ## Overview with description")
except Exception as e:
    print("❌ 2:", e)

# 3 — plan: each deliverable as a checkbox
try:
    plan = generate_project_plan(_SPEC)
    for d in _SPEC.deliverables:
        assert f"- [ ] {d}" in plan, f"deliverable not as checkbox: {d!r}"
    checks += 1; print(f"✅ 3 all {len(_SPEC.deliverables)} deliverables as '- [ ] item'")
except Exception as e:
    print("❌ 3:", e)

# 4 — default_phases: 6 phases with correct names
try:
    phases = default_phases(_SPEC)
    assert len(phases) == 6, f"expected 6 phases, got {len(phases)}"
    names = [p.name for p in phases]
    for expected in ["Plan", "Build", "Test", "Deploy", "Document", "Share"]:
        assert expected in names, f"missing phase '{expected}'"
    checks += 1; print(f"✅ 4 default_phases returns 6 phases: {names}")
except Exception as e:
    print("❌ 4:", e)

# 5 — default_phases: all done=False; each has 3 tasks; Ollama appears in Build
try:
    phases = default_phases(_SPEC)
    assert all(not p.done for p in phases), "all phases should start with done=False"
    assert all(len(p.tasks) == 3 for p in phases), "each phase should have 3 tasks"
    build_tasks = " ".join(next(p.tasks for p in phases if p.name == "Build"))
    assert "Ollama" in build_tasks, f"Ollama should appear in Build tasks: {build_tasks}"
    checks += 1; print("✅ 5 all phases: done=False, 3 tasks; Ollama in Build tasks")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
